# YUYEONG — 오늘의 경험 전체를 설계하는 AI Agent

이 Notebook은 YUYEONG의 **기획 → 수립 → 기술스택·실행 → 수정 → 회고** 전 과정을 코드로 재현한 포트폴리오 문서입니다.

단순 장소 추천이 아니라 위치·시간·동행·분위기·식사 예산·날씨·실제 이동시간을 도구로 확인하고, **식당 4곳 · 카페 4곳 · 20분 이상 드라이브 4곳**을 근거와 함께 제안한 뒤 점심 → 카페 → 저녁 → 드라이브의 하루 동선을 설계합니다.

핵심 원칙:

- 식사 예산은 **점심/저녁 식당 한 곳당 2인 식사금액**에만 적용합니다.
- 카페·드라이브·주유·주차는 식사 예산 점수에 포함하지 않습니다.
- ReAct는 내부 추론을 노출하지 않고 `Action → Observation → Decision/Replan`만 기록합니다.
- 확인되지 않은 가격·메뉴·영업 여부·평점·분위기는 사실처럼 말하지 않습니다.
- 하드 제약은 LLM이 아니라 결정론적 코드와 검증 함수가 보장합니다.


## 0. 실행 가이드

Python 3.10+ 표준 라이브러리만으로 구성했습니다. 외부 서비스 호출은 `RUN_LIVE = True`로 바꿨을 때만 실행됩니다. 실제 키는 출력하지 않으며 `.env`를 Notebook에 직접 삽입하지 않습니다.

OpenAI 호출은 사용자가 지정한 `gpt-5.6-sol`과 Responses API를 사용합니다. 모델 접근 권한과 `OPENAI_API_KEY`가 있어야 활성화됩니다.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Any, Iterable, Literal, Sequence
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, quote
from urllib.request import Request, urlopen
import json
import math
import os
import time

MODEL_ID = "gpt-5.6-sol"
OPENAI_RESPONSES_URL = "https://api.openai.com/v1/responses"
APP_USER_AGENT = "YUYEONG-Portfolio-Notebook/2.0"
RECOMMENDATION_LIMIT = 4
MIN_DRIVE_MINUTES = 20
MAX_REPLANS = 1
RUN_LIVE = False

ROOT = Path.cwd()
if not (ROOT / "package.json").exists() and (ROOT.parent / "package.json").exists():
    ROOT = ROOT.parent

print({
    "project_root": str(ROOT.resolve()),
    "model": MODEL_ID,
    "live_calls": RUN_LIVE,
    "api_key_configured": bool(os.getenv("OPENAI_API_KEY")),
})


## 1. 기획 — 문제 정의와 성공 조건

YUYEONG이 Agent여야 하는 이유는 여러 도구의 관찰 결과가 다음 행동을 바꾸기 때문입니다. 비가 오면 야외 후보를 낮추고, 프리미엄 식당 신호가 부족하면 검색 범위를 한 번 확장하며, 드라이브가 20분 미만이면 후보를 버리고 다시 순위를 계산합니다.


In [ ]:
PRODUCT_REQUIREMENTS = {
    "persona": "연인과 차분하고 분위기 있는 하루를 보내고 싶은 사용자",
    "problem": "장소 목록이 아니라 시간 안에 실행 가능한 전체 경험과 동선이 필요함",
    "inputs": ["출발지", "시작/종료시간", "동행", "분위기", "활동", "식사예산", "취향"],
    "tool_evidence": ["좌표", "날씨", "장소 메타데이터", "도로 이동시간", "도로 거리"],
    "outputs": {
        "restaurants": 4,
        "cafes": 4,
        "drive_courses": "4곳, 직전 지점 기준 20분 이상",
        "itinerary": "점심 → 카페 → 저녁 → 드라이브",
        "menu": "실제 메뉴 URL 또는 명확히 표시된 업종별 가이드",
    },
    "truthfulness": "FACT / INFERENCE / UNKNOWN 분리",
    "portfolio_value": ["도구 오케스트레이션", "ReAct", "검증", "폴백", "추적 가능한 감사 로그"],
}

print(json.dumps(PRODUCT_REQUIREMENTS, ensure_ascii=False, indent=2))


## 2. 수립 — 전체 기술스택과 아키텍처


In [ ]:
TECH_STACK = [
    {"layer": "Language", "technology": "TypeScript 5.9.3", "role": "프로덕션 UI·API의 엄격한 타입"},
    {"layer": "Runtime", "technology": "Node.js >= 22.13", "role": "개발·빌드·테스트 런타임"},
    {"layer": "UI", "technology": "React 19.2.6 / React DOM", "role": "입력, 후보, 메뉴, 동선, ReAct 로그"},
    {"layer": "Framework", "technology": "Next.js 16.2.6 App Router", "role": "페이지와 서버 API Route"},
    {"layer": "Cloud build", "technology": "vinext 0.0.50 + Vite 8.0.13", "role": "Cloudflare 호환 ESM 빌드"},
    {"layer": "Edge", "technology": "Cloudflare Vite Plugin 1.37.1 + Wrangler 4.92", "role": "Worker 개발·배포"},
    {"layer": "Style", "technology": "Tailwind CSS 4.2.1 + CSS", "role": "크고 선명한 반응형 UI"},
    {"layer": "Database", "technology": "Drizzle ORM 0.45.2 / Drizzle Kit", "role": "향후 저장·피드백 확장"},
    {"layer": "LLM", "technology": "OpenAI GPT-5.6 Sol", "role": "검증된 사실의 최종 안전 비평"},
    {"layer": "LLM API", "technology": "OpenAI Responses API", "role": "서버 측 모델 호출"},
    {"layer": "Geocoding", "technology": "OpenStreetMap Nominatim", "role": "장소 문자열 → 좌표"},
    {"layer": "Places", "technology": "OpenStreetMap + Overpass API", "role": "식당·카페·드라이브 후보"},
    {"layer": "Weather", "technology": "Open-Meteo", "role": "기온·강수 조건"},
    {"layer": "Routing", "technology": "OSRM Route + Table API", "role": "동선과 20분 드라이브 검증"},
    {"layer": "Quality", "technology": "ESLint 9.39 + Node test runner", "role": "정적 검사·회귀 테스트"},
    {"layer": "Hosting", "technology": "OpenAI Sites", "role": "비공개 배포·서버 비밀변수"},
    {"layer": "Portfolio", "technology": "Python 3 + Jupyter Notebook", "role": "설계 재현·계약 검증"},
]

for item in TECH_STACK:
    print(f"{item['layer']:12} | {item['technology']:48} | {item['role']}")


In [ ]:
ARCHITECTURE = {
    "client": "React ExperienceAgent.tsx",
    "server": "Next.js App Router POST /api/experience",
    "agent": ["REQUEST", "GROUND", "FILTER", "PLAN", "VERIFY", "REPORT"],
    "react_actions": ["Location", "Weather", "Places", "PremiumExpand", "CandidateExpand", "Rank", "DriveConstraint", "Route", "Verify", "ModelCritic"],
    "external_tools": {
        "Nominatim": "geocoding",
        "Open-Meteo": "weather",
        "Overpass": "places",
        "OSRM": "road routes",
        "OpenAI Responses": "final bounded critic",
    },
    "deployment": "vinext → Vite → Cloudflare Worker → OpenAI Sites",
}

print(json.dumps(ARCHITECTURE, ensure_ascii=False, indent=2))


### 실제 `package.json` 버전과 기술스택 자동 대조


In [ ]:
package = json.loads((ROOT / "package.json").read_text(encoding="utf-8"))
expected_packages = {
    "typescript": "5.9.3",
    "react": "19.2.6",
    "react-dom": "19.2.6",
    "next": "16.2.6",
    "vinext": "0.0.50",
    "vite": "8.0.13",
    "@cloudflare/vite-plugin": "1.37.1",
    "wrangler": "4.92.0",
    "tailwindcss": "4.2.1",
    "drizzle-orm": "0.45.2",
    "drizzle-kit": "0.31.10",
    "eslint": "9.39.4",
}
installed = {**package.get("dependencies", {}), **package.get("devDependencies", {})}
version_audit = {name: installed.get(name) == version for name, version in expected_packages.items()}
for name, passed in version_audit.items():
    print(f"{'PASS' if passed else 'CHECK':5} {name:30} expected={expected_packages[name]} actual={installed.get(name)}")
assert all(version_audit.values())


## 3. 실행 기반 — 도메인 모델과 사용자 요청 정규화


In [ ]:
ActivityKind = Literal["lunch", "cafe", "dinner", "drive"]
Transport = Literal["drive", "walk"]
BudgetTier = Literal["light", "date", "special", "signature"]

@dataclass(frozen=True)
class Coordinates:
    latitude: float
    longitude: float

@dataclass
class ExperienceRequest:
    query: str
    start_time: str = "11:30"
    end_time: str = "21:30"
    companion: str = "연인과"
    mood: str = "분위기 있게"
    transport: Transport = "drive"
    activities: list[ActivityKind] = field(default_factory=lambda: ["lunch", "cafe", "dinner", "drive"])
    preferences: list[str] = field(default_factory=list)
    budget_tier: BudgetTier = "special"

@dataclass
class Place:
    id: str
    kind: Literal["restaurant", "cafe", "drive"]
    name: str
    coordinates: Coordinates
    cuisine_key: str = "restaurant"
    cuisine: str = "일반 음식점"
    place_type: str = "장소"
    opening_hours: str | None = None
    menu_url: str | None = None
    website_url: str | None = None
    phone: str | None = None
    premium_score: int = 0
    premium_signals: list[str] = field(default_factory=list)

@dataclass
class Weather:
    temperature: float
    apparent_temperature: float
    precipitation: float
    rain_probability: float | None
    description: str
    rainy: bool

@dataclass
class TravelLeg:
    origin: str
    destination: str
    minutes: int
    distance_meters: int
    source: Literal["OSRM", "estimated"]

@dataclass
class Recommendation:
    place: Place
    travel_minutes: int | None = None
    travel_source: str | None = None
    evidence: list[str] = field(default_factory=list)
    unknowns: list[str] = field(default_factory=list)

@dataclass
class ReActCycle:
    id: str
    action: dict[str, Any]
    observation: dict[str, Any]
    decision: str
    replan: bool = False

@dataclass
class ExperienceResult:
    request: ExperienceRequest
    location_label: str
    coordinates: Coordinates
    weather: Weather | None
    restaurants: list[Recommendation]
    cafes: list[Recommendation]
    drives: list[Recommendation]
    itinerary: list[dict[str, Any]]
    route_legs: list[TravelLeg]
    react_trace: list[ReActCycle]
    six_stage_audit: list[dict[str, Any]]
    model_audit: dict[str, Any]
    unknowns: list[str]


In [ ]:
BUDGET_PROFILES = {
    "light": {
        "label": "가벼운 식사", "range": "식당 1곳 · 2인 6만원 이하",
        "premium_weight": -0.12, "distance_rate": 0.035,
    },
    "date": {
        "label": "설레는 식사", "range": "식당 1곳 · 2인 6–14만원",
        "premium_weight": 0.34, "distance_rate": 0.030,
    },
    "special": {
        "label": "특별한 식사", "range": "식당 1곳 · 2인 14–30만원",
        "premium_weight": 0.86, "distance_rate": 0.022,
    },
    "signature": {
        "label": "시그니처 식사", "range": "식당 1곳 · 2인 30만원 이상",
        "premium_weight": 1.34, "distance_rate": 0.014,
    },
}

def hhmm_to_minutes(value: str) -> int:
    parsed = datetime.strptime(value, "%H:%M")
    return parsed.hour * 60 + parsed.minute

def normalize_request(raw: dict[str, Any]) -> ExperienceRequest:
    request = ExperienceRequest(
        query=str(raw.get("query", "")).strip(),
        start_time=str(raw.get("start_time", "11:30")),
        end_time=str(raw.get("end_time", "21:30")),
        companion=str(raw.get("companion", "연인과")),
        mood=str(raw.get("mood", "분위기 있게")),
        transport=raw.get("transport", "drive"),
        activities=list(dict.fromkeys(raw.get("activities") or ["lunch", "cafe", "dinner", "drive"])),
        preferences=[str(item).strip() for item in raw.get("preferences", []) if str(item).strip()],
        budget_tier=raw.get("budget_tier", "special"),
    )
    if not request.query:
        raise ValueError("출발 지역이 필요합니다.")
    if request.transport not in {"drive", "walk"}:
        raise ValueError("transport는 drive 또는 walk여야 합니다.")
    if request.budget_tier not in BUDGET_PROFILES:
        raise ValueError("지원하지 않는 식사 예산 단계입니다.")
    allowed = {"lunch", "cafe", "dinner", "drive"}
    if not request.activities or not set(request.activities) <= allowed:
        raise ValueError("지원하지 않는 활동이 포함되어 있습니다.")
    if hhmm_to_minutes(request.end_time) <= hhmm_to_minutes(request.start_time):
        raise ValueError("종료시간은 시작시간보다 늦어야 합니다.")
    return request

sample_request = normalize_request({
    "query": "성수역, 서울",
    "start_time": "11:30",
    "end_time": "21:30",
    "companion": "연인과",
    "mood": "분위기 있게",
    "activities": ["lunch", "cafe", "dinner", "drive"],
    "budget_tier": "special",
    "preferences": ["일식", "이탈리안", "조용한 카페"],
})
print(asdict(sample_request))
print("식사예산:", BUDGET_PROFILES[sample_request.budget_tier])


## 4. 도구 계층 — 외부 사실을 가져오는 코드


In [ ]:
class ToolError(RuntimeError):
    pass

def http_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
    method: str = "GET",
    payload: dict[str, Any] | None = None,
    headers: dict[str, str] | None = None,
    timeout: int = 20,
) -> dict[str, Any] | list[Any]:
    if params:
        url = f"{url}?{urlencode(params)}"
    body = json.dumps(payload).encode("utf-8") if payload is not None else None
    request_headers = {
        "User-Agent": APP_USER_AGENT,
        "Accept": "application/json",
        **({"Content-Type": "application/json"} if payload is not None else {}),
        **(headers or {}),
    }
    request = Request(url, data=body, headers=request_headers, method=method)
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
        raise ToolError(f"도구 호출 실패: {url.split('?')[0]} ({type(exc).__name__})") from exc

print("HTTP tool ready — API key 값은 코드나 로그에 포함하지 않습니다.")


In [ ]:
_last_nominatim_call = 0.0

def geocode_nominatim(query: str) -> tuple[Coordinates, str]:
    global _last_nominatim_call
    wait_seconds = max(0.0, 1.1 - (time.monotonic() - _last_nominatim_call))
    if wait_seconds:
        time.sleep(wait_seconds)
    data = http_json(
        "https://nominatim.openstreetmap.org/search",
        params={"q": query, "format": "jsonv2", "limit": 1, "accept-language": "ko"},
    )
    _last_nominatim_call = time.monotonic()
    if not isinstance(data, list) or not data:
        raise ToolError("출발 지역을 좌표로 찾지 못했습니다.")
    item = data[0]
    return Coordinates(float(item["lat"]), float(item["lon"])), str(item.get("display_name", query))

def fetch_weather_open_meteo(point: Coordinates) -> Weather:
    data = http_json(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": point.latitude,
            "longitude": point.longitude,
            "current": "temperature_2m,apparent_temperature,precipitation,weather_code",
            "hourly": "precipitation_probability",
            "timezone": "Asia/Seoul",
            "forecast_days": 1,
        },
    )
    current = data.get("current", {})
    probabilities = data.get("hourly", {}).get("precipitation_probability", [])
    rain_probability = max(probabilities) if probabilities else None
    precipitation = float(current.get("precipitation", 0) or 0)
    code = int(current.get("weather_code", -1))
    rainy = precipitation > 0 or (rain_probability or 0) >= 50 or code in {51, 53, 55, 61, 63, 65, 80, 81, 82}
    return Weather(
        temperature=float(current.get("temperature_2m", 0)),
        apparent_temperature=float(current.get("apparent_temperature", 0)),
        precipitation=precipitation,
        rain_probability=rain_probability,
        description="비 가능성 있음" if rainy else "야외 이동 가능",
        rainy=rainy,
    )

print("Nominatim geocoder + Open-Meteo weather tools ready")


In [ ]:
CUISINE_LABELS = {
    "korean": "한식", "japanese": "일식", "sushi": "초밥", "ramen": "라멘",
    "chinese": "중식", "italian": "이탈리안", "french": "프렌치", "fusion": "퓨전",
    "steak_house": "스테이크하우스", "seafood": "해산물", "thai": "태국 음식",
    "indian": "인도 음식", "mexican": "멕시칸", "vietnamese": "베트남 음식",
    "restaurant": "일반 음식점", "fast_food": "간편식", "food_court": "푸드코트",
}

def overpass_query(center: Coordinates, radius_m: int = 10000, premium: bool = False) -> str:
    lat, lon = center.latitude, center.longitude
    restaurant_filter = '["amenity"~"restaurant|fast_food|food_court"]'
    cafe_filter = '["amenity"="cafe"]'
    scenic_filters = [
        '["tourism"="viewpoint"]', '["tourism"="attraction"]',
        '["leisure"="park"]', '["leisure"="garden"]',
    ]
    blocks = []
    for element in ("node", "way", "relation"):
        blocks.append(f'{element}(around:{radius_m},{lat},{lon}){restaurant_filter};')
        blocks.append(f'{element}(around:{radius_m},{lat},{lon}){cafe_filter};')
        for item_filter in scenic_filters:
            blocks.append(f'{element}(around:{radius_m},{lat},{lon}){item_filter};')
    if premium:
        premium_filters = ['["cuisine"~"sushi|japanese|french|italian|steak_house",i]', '["reservation"]', '["menu"]']
        for element in ("node", "way", "relation"):
            for item_filter in premium_filters:
                blocks.append(f'{element}(around:{radius_m},{lat},{lon})["amenity"="restaurant"]{item_filter};')
    return "[out:json][timeout:25];(" + "".join(blocks) + ");out center tags;"

def fetch_places_overpass(center: Coordinates, radius_m: int = 10000, premium: bool = False) -> list[dict[str, Any]]:
    data = http_json(
        "https://overpass-api.de/api/interpreter",
        method="POST",
        payload=None,
        params={"data": overpass_query(center, radius_m, premium)},
        timeout=30,
    )
    return list(data.get("elements", []))

print(overpass_query(Coordinates(37.5445, 127.0560), radius_m=8000)[:420] + "...")


In [ ]:
PREMIUM_KEYWORDS = ("omakase", "오마카세", "course", "코스", "fine", "파인", "chef", "셰프", "tasting")

def safe_http_url(value: str | None) -> str | None:
    if value and value.startswith(("https://", "http://")):
        return value
    return None

def premium_profile(tags: dict[str, str], cuisine_key: str) -> tuple[int, list[str]]:
    searchable = " ".join(str(value).lower() for value in tags.values())
    signals: list[str] = []
    score = 0
    if cuisine_key in {"sushi", "french", "steak_house"}:
        score += 22
        signals.append("프리미엄 탐색 업종 신호")
    if any(keyword in searchable for keyword in PREMIUM_KEYWORDS):
        score += 34
        signals.append("코스·셰프 관련 등록 메타데이터")
    if tags.get("reservation") in {"yes", "required", "recommended"}:
        score += 18
        signals.append("예약 관련 등록 메타데이터")
    if tags.get("menu") or tags.get("contact:menu"):
        score += 12
        signals.append("메뉴 링크 등록")
    if tags.get("website") or tags.get("contact:website"):
        score += 8
        signals.append("웹사이트 등록")
    return min(100, score), signals[:4]

def osm_to_place(element: dict[str, Any]) -> Place | None:
    tags = element.get("tags") or {}
    center = element.get("center") or {}
    lat = element.get("lat", center.get("lat"))
    lon = element.get("lon", center.get("lon"))
    name = str(tags.get("name", "")).strip()
    if not name or lat is None or lon is None:
        return None
    amenity = tags.get("amenity")
    if amenity == "cafe":
        kind = "cafe"
    elif amenity in {"restaurant", "fast_food", "food_court"}:
        kind = "restaurant"
    else:
        kind = "drive"
    cuisine_key = str(tags.get("cuisine", amenity or "restaurant")).split(";")[0]
    cuisine = CUISINE_LABELS.get(cuisine_key, cuisine_key.replace("_", " "))
    place_type = (
        "전망 포인트" if tags.get("tourism") == "viewpoint" else
        "정원" if tags.get("leisure") == "garden" else
        "공원" if tags.get("leisure") == "park" else
        "명소" if tags.get("tourism") == "attraction" else cuisine
    )
    premium_score, premium_signals = premium_profile(tags, cuisine_key) if kind == "restaurant" else (0, [])
    return Place(
        id=f"{element.get('type', 'osm')}-{element.get('id')}",
        kind=kind,
        name=name,
        coordinates=Coordinates(float(lat), float(lon)),
        cuisine_key=cuisine_key,
        cuisine=cuisine,
        place_type=place_type,
        opening_hours=tags.get("opening_hours"),
        menu_url=safe_http_url(tags.get("menu") or tags.get("contact:menu")),
        website_url=safe_http_url(tags.get("website") or tags.get("contact:website")),
        phone=tags.get("phone") or tags.get("contact:phone"),
        premium_score=premium_score,
        premium_signals=premium_signals,
    )

def normalize_places(elements: Iterable[dict[str, Any]]) -> list[Place]:
    result: list[Place] = []
    seen: set[tuple[str, float, float]] = set()
    for element in elements:
        place = osm_to_place(element)
        if place is None:
            continue
        key = (place.name.casefold(), round(place.coordinates.latitude, 4), round(place.coordinates.longitude, 4))
        if key not in seen:
            seen.add(key)
            result.append(place)
    return result

print("OSM normalization + premium-signal extraction ready")


### OSRM 도로 경로와 20분 드라이브 제약


In [ ]:
def haversine_meters(a: Coordinates, b: Coordinates) -> float:
    earth = 6_371_000
    lat1, lat2 = math.radians(a.latitude), math.radians(b.latitude)
    dlat = lat2 - lat1
    dlon = math.radians(b.longitude - a.longitude)
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return earth * 2 * math.atan2(math.sqrt(h), math.sqrt(1 - h))

def estimated_leg(origin_name: str, destination: Place, origin: Coordinates, transport: Transport = "drive") -> TravelLeg:
    factor = 1.18 if transport == "walk" else 1.30
    speed_mps = 1.25 if transport == "walk" else 7.20
    distance = haversine_meters(origin, destination.coordinates) * factor
    return TravelLeg(origin_name, destination.name, max(1, round(distance / speed_mps / 60)), round(distance), "estimated")

def osrm_route(points: Sequence[tuple[str, Coordinates]], transport: Transport = "drive") -> list[TravelLeg]:
    if len(points) < 2:
        return []
    profile = "foot" if transport == "walk" else "driving"
    coordinates = ";".join(f"{p.longitude:.6f},{p.latitude:.6f}" for _, p in points)
    try:
        data = http_json(
            f"https://router.project-osrm.org/route/v1/{profile}/{coordinates}",
            params={"overview": "false", "steps": "false"},
            timeout=15,
        )
        route = data["routes"][0]
        return [
            TravelLeg(points[i][0], points[i + 1][0], max(1, round(leg["duration"] / 60)), round(leg["distance"]), "OSRM")
            for i, leg in enumerate(route["legs"])
        ]
    except (ToolError, KeyError, IndexError):
        return [
            estimated_leg(points[i][0], Place("fallback", "drive", points[i + 1][0], points[i + 1][1]), points[i][1], transport)
            for i in range(len(points) - 1)
        ]

def osrm_drive_table(origin: Coordinates, candidates: Sequence[Place]) -> list[Recommendation]:
    pool = list(candidates)[:24]
    if not pool:
        return []
    coordinates = ";".join(
        f"{point.longitude:.6f},{point.latitude:.6f}"
        for point in [origin, *[place.coordinates for place in pool]]
    )
    try:
        data = http_json(
            f"https://router.project-osrm.org/table/v1/driving/{coordinates}",
            params={"sources": 0, "annotations": "duration,distance"},
            timeout=18,
        )
        durations = data["durations"][0]
        distances = data["distances"][0]
        ranked = [
            Recommendation(
                place=place,
                travel_minutes=max(1, round(durations[i + 1] / 60)),
                travel_source="OSRM",
                evidence=[f"OSRM 도로 이동 {max(1, round(durations[i + 1] / 60))}분", f"도로 거리 {round(distances[i + 1])}m"],
            )
            for i, place in enumerate(pool)
            if durations[i + 1] is not None and durations[i + 1] >= MIN_DRIVE_MINUTES * 60
        ]
    except (ToolError, KeyError, IndexError, TypeError):
        ranked = []
        for place in pool:
            leg = estimated_leg("직전 일정", place, origin, "drive")
            if leg.minutes >= MIN_DRIVE_MINUTES:
                ranked.append(Recommendation(
                    place=place,
                    travel_minutes=leg.minutes,
                    travel_source="estimated",
                    evidence=[f"직선거리 기반 추정 {leg.minutes}분"],
                    unknowns=["실제 도로 시간은 OSRM 재확인 필요"],
                ))
    scenic = {"전망 포인트": 4, "정원": 3, "공원": 2, "명소": 1}
    return sorted(ranked, key=lambda item: (scenic.get(item.place.place_type, 0), -abs((item.travel_minutes or 0) - 35)), reverse=True)

print("OSRM Route/Table + estimated fallback ready")


## 5. 판단 계층 — 예산, 분위기, 다양성, 메뉴 표시


In [ ]:
MOOD_CUISINE = {
    "편안하게": ["korean", "japanese", "noodle", "restaurant"],
    "분위기 있게": ["italian", "japanese", "sushi", "french"],
    "새롭게": ["thai", "indian", "mexican", "vietnamese"],
    "든든하게": ["korean", "barbecue", "chinese", "steak_house"],
    "가볍게": ["salad", "sandwich", "sushi", "vietnamese"],
}

def restaurant_score(place: Place, request: ExperienceRequest, origin: Coordinates) -> float:
    budget = BUDGET_PROFILES[request.budget_tier]
    mood_bonus = 34 if place.cuisine_key in MOOD_CUISINE.get(request.mood, []) else 0
    preference_bonus = 18 if place.cuisine in request.preferences or place.cuisine_key in request.preferences else 0
    premium_bonus = place.premium_score * budget["premium_weight"]
    distance_penalty = haversine_meters(origin, place.coordinates) * budget["distance_rate"]
    return mood_bonus + preference_bonus + premium_bonus - distance_penalty

def cafe_score(place: Place, origin: Coordinates, rainy: bool) -> float:
    # 중요: 식사 예산을 인자로 받지 않습니다.
    distance_penalty = haversine_meters(origin, place.coordinates) * 0.022
    metadata_bonus = 10 if place.website_url else 0
    rain_bonus = 5 if rainy and place.opening_hours else 0
    return metadata_bonus + rain_bonus - distance_penalty

def rank_restaurants(places: Sequence[Place], request: ExperienceRequest, origin: Coordinates) -> list[Recommendation]:
    ranked = sorted(
        (place for place in places if place.kind == "restaurant"),
        key=lambda place: restaurant_score(place, request, origin),
        reverse=True,
    )
    return [
        Recommendation(
            place=place,
            evidence=[
                f"식사 예산 탐색 단계: {BUDGET_PROFILES[request.budget_tier]['label']}",
                *place.premium_signals,
            ],
            unknowns=["실제 2인 결제금액", "현재 예약 가능 여부"],
        )
        for place in ranked[:RECOMMENDATION_LIMIT]
    ]

def rank_cafes(places: Sequence[Place], origin: Coordinates, rainy: bool) -> list[Recommendation]:
    ranked = sorted(
        (place for place in places if place.kind == "cafe"),
        key=lambda place: cafe_score(place, origin, rainy),
        reverse=True,
    )
    return [
        Recommendation(place=place, evidence=["카페 순위에는 식사 예산 미적용"], unknowns=["현재 좌석·영업 여부"])
        for place in ranked[:RECOMMENDATION_LIMIT]
    ]

print("Restaurant budget and budget-independent cafe rankers ready")


In [ ]:
MENU_GUIDES = {
    "korean": ["찌개·전골", "구이·정식", "비빔밥·덮밥"],
    "japanese": ["돈카츠·덮밥", "우동·소바", "초밥·사시미"],
    "sushi": ["모둠초밥", "사시미", "회덮밥"],
    "italian": ["파스타", "리조또", "스테이크"],
    "french": ["에피타이저", "메인 요리", "디저트"],
    "steak_house": ["스테이크", "사이드", "와인"],
    "restaurant": ["메인 요리", "식사 메뉴", "사이드"],
}

def menu_view(place: Place) -> dict[str, Any]:
    if place.menu_url:
        return {
            "status": "registered_menu_link",
            "url": place.menu_url,
            "items": [],
            "notice": "OpenStreetMap에 등록된 메뉴 링크입니다. 방문 전 최신 정보 확인이 필요합니다.",
        }
    return {
        "status": "category_guide",
        "url": None,
        "items": MENU_GUIDES.get(place.cuisine_key, MENU_GUIDES["restaurant"]),
        "notice": "실제 매장 메뉴가 아닌 업종별 탐색 가이드입니다.",
    }

example_place = Place("demo", "restaurant", "예시 식당", Coordinates(37.5, 127.0), cuisine_key="italian", cuisine="이탈리안")
print(json.dumps(menu_view(example_place), ensure_ascii=False, indent=2))


## 6. Agent의 6단계 사고 시스템 + Bounded ReAct


In [ ]:
SIX_STAGES = [
    ("01", "REQUEST", "의도 정렬", "입력과 하드 제약 정규화"),
    ("02", "GROUND", "사실 수집", "위치·날씨·장소·경로 도구 호출"),
    ("03", "FILTER", "후보 검증", "이름·좌표·중복·개수 확인"),
    ("04", "PLAN", "동선 설계", "점심·카페·저녁·드라이브 순서와 시간 배치"),
    ("05", "VERIFY", "반증·검산", "예산 범위·20분·중복·시간창·출처 검증"),
    ("06", "REPORT", "안전한 응답", "FACT·INFERENCE·UNKNOWN과 근거 반환"),
]

class TraceRecorder:
    def __init__(self) -> None:
        self.cycles: list[ReActCycle] = []

    def record(
        self,
        tool: str,
        label: str,
        input_summary: str,
        observation_summary: str,
        facts: Sequence[str],
        decision: str,
        *,
        status: Literal["grounded", "attention"] = "grounded",
        replan: bool = False,
    ) -> None:
        self.cycles.append(ReActCycle(
            id=f"{len(self.cycles) + 1:02d}",
            action={"tool": tool, "label": label, "input": input_summary},
            observation={"status": status, "summary": observation_summary, "facts": list(facts)},
            decision=decision,
            replan=replan,
        ))

recorder = TraceRecorder()
recorder.record("Location", "출발지 확인", "성수역, 서울", "좌표 확인", ["37.5445, 127.0560"], "주변 장소 탐색을 시작합니다.")
print(json.dumps([asdict(cycle) for cycle in recorder.cycles], ensure_ascii=False, indent=2))


### Chain-of-Thought 안전 원칙

Notebook과 제품 UI에는 모델의 비공개 추론을 저장하거나 노출하지 않습니다. 아래 코드처럼 도구 이름, 입력 요약, 관찰된 사실, 공개 가능한 결정, 재계획 여부만 남깁니다. 이는 추적 가능성을 확보하면서도 원시 사고의 사슬을 요구하지 않는 구조입니다.


In [ ]:
ACTIVITY_DURATION = {"lunch": 75, "cafe": 70, "dinner": 95, "drive": 90}

def build_itinerary(
    request: ExperienceRequest,
    restaurants: Sequence[Recommendation],
    cafes: Sequence[Recommendation],
    drives: Sequence[Recommendation],
) -> list[dict[str, Any]]:
    if len(restaurants) < 2 and {"lunch", "dinner"} <= set(request.activities):
        raise ValueError("점심과 저녁에 사용할 서로 다른 식당 2곳이 필요합니다.")
    selected = {
        "lunch": restaurants[0] if restaurants else None,
        "cafe": cafes[0] if cafes else None,
        "dinner": restaurants[1] if len(restaurants) > 1 else (restaurants[0] if restaurants else None),
        "drive": drives[0] if drives else None,
    }
    start = datetime.strptime(request.start_time, "%H:%M")
    end_limit = datetime.strptime(request.end_time, "%H:%M")
    order = [activity for activity in ("lunch", "cafe", "dinner", "drive") if activity in request.activities]
    itinerary: list[dict[str, Any]] = []
    cursor = start
    for activity in order:
        item = selected[activity]
        if item is None:
            raise ValueError(f"{activity} 후보가 없습니다.")
        finish = cursor + timedelta(minutes=ACTIVITY_DURATION[activity])
        itinerary.append({
            "activity": activity,
            "start": cursor.strftime("%H:%M"),
            "end": finish.strftime("%H:%M"),
            "place": item.place.name,
            "coordinates": asdict(item.place.coordinates),
            "menu": menu_view(item.place) if activity in {"lunch", "dinner"} else None,
        })
        cursor = finish
    if cursor > end_limit:
        raise ValueError("활동 시간이 사용자 종료시간을 초과합니다. 이동시간을 포함해 재계획해야 합니다.")
    return itinerary

print("Itinerary builder ready")


## 7. 검증 계층 — 환각 방지용 결정론적 하드 체크


In [ ]:
def verify_result(
    restaurants: Sequence[Recommendation],
    cafes: Sequence[Recommendation],
    drives: Sequence[Recommendation],
    itinerary: Sequence[dict[str, Any]],
    request: ExperienceRequest,
) -> list[dict[str, Any]]:
    all_ids = [item.place.id for item in [*restaurants, *cafes, *drives]]
    checks = [
        {"name": "식당 후보 4곳", "passed": len(restaurants) == RECOMMENDATION_LIMIT},
        {"name": "카페 후보 4곳", "passed": len(cafes) == RECOMMENDATION_LIMIT},
        {"name": "드라이브 후보 4곳", "passed": len(drives) == RECOMMENDATION_LIMIT},
        {"name": "드라이브 모두 20분 이상", "passed": all((item.travel_minutes or 0) >= MIN_DRIVE_MINUTES for item in drives)},
        {"name": "후보 중복 없음", "passed": len(all_ids) == len(set(all_ids))},
        {"name": "요청 활동 모두 포함", "passed": set(request.activities) <= {item["activity"] for item in itinerary}},
        {"name": "점심·저녁 서로 다른 식당", "passed": not ({"lunch", "dinner"} <= set(request.activities)) or itinerary[0]["place"] != next(item["place"] for item in itinerary if item["activity"] == "dinner")},
        {"name": "식사 예산은 식당에만 적용", "passed": all("식사 예산 탐색 단계" not in " ".join(item.evidence) for item in [*cafes, *drives])},
        {"name": "실제 메뉴와 가이드 구분", "passed": all((item.get("menu") or {}).get("status") in {"registered_menu_link", "category_guide"} for item in itinerary if item["activity"] in {"lunch", "dinner"})},
    ]
    failed = [check["name"] for check in checks if not check["passed"]]
    if failed:
        raise ValueError("하드 제약 실패: " + ", ".join(failed))
    return checks

print("Deterministic verifier ready")


## 8. GPT-5.6 Sol — 장소를 만들지 않는 최종 Model Critic


In [ ]:
MODEL_INSTRUCTIONS = (
    "You are the final safety critic for YUYEONG. "
    "Use only the supplied JSON facts. Verify that each requested recommendation category has exactly four candidates "
    "and every drive recommendation is at least minimumDriveMinutes. Do not add venue claims, prices, ratings, "
    "ambience, menus, opening status, or reservation availability. Do not reveal chain-of-thought. "
    "Return one concise Korean sentence stating whether the constraints are internally consistent and name the most important unknown."
)

def extract_response_text(data: dict[str, Any]) -> str:
    if isinstance(data.get("output_text"), str):
        return data["output_text"].strip()
    chunks: list[str] = []
    for output in data.get("output", []):
        for content in output.get("content", []):
            if content.get("type") == "output_text" and isinstance(content.get("text"), str):
                chunks.append(content["text"].strip())
    return " ".join(filter(None, chunks))

def run_model_critic(audit_input: dict[str, Any]) -> dict[str, Any]:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return {
            "id": MODEL_ID,
            "status": "fallback",
            "used": False,
            "summary": "API 키가 없어 결정론적 검증 결과를 유지합니다.",
            "note": "키 값은 Notebook에 저장하거나 출력하지 않습니다.",
        }
    try:
        response = http_json(
            OPENAI_RESPONSES_URL,
            method="POST",
            headers={"Authorization": f"Bearer {api_key}"},
            payload={
                "model": MODEL_ID,
                "reasoning": {"effort": "medium"},
                "max_output_tokens": 180,
                "instructions": MODEL_INSTRUCTIONS,
                "input": json.dumps(audit_input, ensure_ascii=False),
            },
            timeout=30,
        )
        summary = extract_response_text(response)[:360]
        if not summary:
            raise ToolError("빈 모델 응답")
        return {"id": MODEL_ID, "status": "active", "used": True, "summary": summary}
    except ToolError:
        return {
            "id": MODEL_ID,
            "status": "fallback",
            "used": False,
            "summary": "모델 호출 실패를 숨기지 않고 결정론적 검증 결과를 유지합니다.",
        }

print(run_model_critic({"dry_run": True, "minimumDriveMinutes": MIN_DRIVE_MINUTES}))


## 9. 전체 Agent 오케스트레이터


In [ ]:
def ensure_exact_four(items: Sequence[Recommendation], category: str) -> list[Recommendation]:
    unique: list[Recommendation] = []
    seen: set[str] = set()
    for item in items:
        if item.place.id not in seen:
            seen.add(item.place.id)
            unique.append(item)
    if len(unique) < RECOMMENDATION_LIMIT:
        raise ValueError(f"{category} 후보가 {RECOMMENDATION_LIMIT}곳보다 적습니다. 검색 범위를 조정하세요.")
    return unique[:RECOMMENDATION_LIMIT]

def build_stage_audit(request: ExperienceRequest, checks: Sequence[dict[str, Any]], unknowns: Sequence[str]) -> list[dict[str, Any]]:
    detail = {
        "REQUEST": f"{request.query} · {request.start_time}–{request.end_time} · {request.budget_tier}",
        "GROUND": "Nominatim / Open-Meteo / Overpass / OSRM 출처 기록",
        "FILTER": "이름·좌표·중복과 후보 개수 확인",
        "PLAN": "점심 → 카페 → 저녁 → 드라이브 순서 설계",
        "VERIFY": f"하드 체크 {sum(check['passed'] for check in checks)}/{len(checks)} 통과",
        "REPORT": f"UNKNOWN {len(unknowns)}건을 사실과 분리",
    }
    return [{"id": sid, "key": key, "name": name, "purpose": purpose, "result": detail[key]} for sid, key, name, purpose in SIX_STAGES]

def run_yuyeong_agent(request: ExperienceRequest) -> ExperienceResult:
    if not RUN_LIVE:
        raise RuntimeError("실제 외부 도구 실행은 RUN_LIVE=True로 명시적으로 활성화하세요.")

    trace = TraceRecorder()
    coordinates, location_label = geocode_nominatim(request.query)
    trace.record("Location", "출발지 확인", request.query, location_label, [f"{coordinates.latitude:.5f}, {coordinates.longitude:.5f}"], "날씨와 장소를 조회합니다.")

    weather: Weather | None
    try:
        weather = fetch_weather_open_meteo(coordinates)
        trace.record("Weather", "현재 날씨", location_label, weather.description, [f"기온 {weather.temperature:.1f}°C", f"강수확률 {weather.rain_probability}%"], "장소 순위에 날씨를 반영합니다.")
    except ToolError:
        weather = None
        trace.record("Weather", "현재 날씨", location_label, "조회 실패", [], "날씨를 UNKNOWN으로 두고 계속합니다.", status="attention")

    elements = fetch_places_overpass(coordinates, radius_m=10000)
    places = normalize_places(elements)
    trace.record("Places", "주변 후보 탐색", "반경 10km", f"정규화 후보 {len(places)}곳", ["OpenStreetMap / Overpass"], "카테고리별 순위를 계산합니다.")

    restaurants = rank_restaurants(places, request, coordinates)
    needs_expansion = len(restaurants) < 4 or (
        request.budget_tier in {"special", "signature"} and any(item.place.premium_score < 28 for item in restaurants)
    )
    if needs_expansion:
        expanded = normalize_places(fetch_places_overpass(coordinates, radius_m=18000, premium=True))
        merged = normalize_places([*elements, *fetch_places_overpass(coordinates, radius_m=18000, premium=True)])
        places = merged
        restaurants = rank_restaurants(places, request, coordinates)
        trace.record("PremiumExpand", "프리미엄 후보 확장", "반경 18km · 최대 1회", f"추가 후보 {len(expanded)}곳", ["검색 조건 변경"], "확장 결과로 다시 순위를 계산합니다.", replan=True)

    cafes = rank_cafes(places, coordinates, bool(weather and weather.rainy))
    drive_places = [place for place in places if place.kind == "drive"]
    drives = osrm_drive_table(coordinates, drive_places)
    trace.record("DriveConstraint", "20분 드라이브 검증", f"후보 {len(drive_places)}곳", f"통과 {len(drives)}곳", ["최소 20분"], "짧은 후보를 제거합니다.")

    restaurants = ensure_exact_four(restaurants, "식당")
    cafes = ensure_exact_four(cafes, "카페")
    drives = ensure_exact_four(drives, "드라이브")
    trace.record("Rank", "최종 후보 선정", request.mood, "식당 4 · 카페 4 · 드라이브 4", ["중복 제거", "식당 예산만 적용"], "동선 초안을 생성합니다.")

    itinerary = build_itinerary(request, restaurants, cafes, drives)
    route_points = [(location_label, coordinates)] + [
        (item["place"], Coordinates(**item["coordinates"])) for item in itinerary
    ]
    route_legs = osrm_route(route_points, request.transport)
    trace.record("Route", "전체 동선 계산", f"{len(route_points)}개 지점", f"{len(route_legs)}개 구간", [f"총 이동 {sum(leg.minutes for leg in route_legs)}분"], "최종 하드 체크를 실행합니다.")

    checks = verify_result(restaurants, cafes, drives, itinerary, request)
    unknowns = sorted({unknown for item in [*restaurants, *cafes, *drives] for unknown in item.unknowns})
    trace.record("Verify", "반증·검산", "전체 계획", f"{len(checks)}개 하드 체크 통과", [check["name"] for check in checks], "검증된 결과만 보고합니다.")

    audit_input = {
        "recommendationCounts": {"restaurants": len(restaurants), "cafes": len(cafes), "drives": len(drives)},
        "minimumDriveMinutes": MIN_DRIVE_MINUTES,
        "driveRecommendations": [{"name": item.place.name, "minutes": item.travel_minutes, "source": item.travel_source} for item in drives],
        "hardChecks": checks,
        "unknowns": unknowns,
    }
    model_audit = run_model_critic(audit_input)
    trace.record("ModelCritic", "최종 안전 요약", MODEL_ID, model_audit["status"], [model_audit["summary"]], "도구 기반 선택은 변경하지 않습니다.", status="grounded" if model_audit["used"] else "attention")

    return ExperienceResult(
        request=request,
        location_label=location_label,
        coordinates=coordinates,
        weather=weather,
        restaurants=restaurants,
        cafes=cafes,
        drives=drives,
        itinerary=itinerary,
        route_legs=route_legs,
        react_trace=trace.cycles,
        six_stage_audit=build_stage_audit(request, checks, unknowns),
        model_audit=model_audit,
        unknowns=unknowns,
    )

print("YUYEONG orchestrator assembled")


### 선택적 라이브 실행 — 기본값은 네트워크 호출 없음


In [ ]:
if RUN_LIVE:
    live_result = run_yuyeong_agent(sample_request)
    print(json.dumps(asdict(live_result), ensure_ascii=False, indent=2))
else:
    print("SKIP: RUN_LIVE=False — 포트폴리오 검증 셀은 아래 fixture로 계속됩니다.")


## 10. API 키 없이 실행하는 결정론적 통합 테스트


In [ ]:
def fixture_place(index: int, kind: Literal["restaurant", "cafe", "drive"], *, premium: int = 0) -> Place:
    cuisine_cycle = ["japanese", "italian", "sushi", "french"]
    cuisine_key = cuisine_cycle[index % len(cuisine_cycle)] if kind == "restaurant" else "coffee_shop"
    return Place(
        id=f"{kind}-{index}",
        kind=kind,
        name=f"테스트 {kind} {index}",
        coordinates=Coordinates(37.54 + index * 0.006, 127.05 + index * 0.006),
        cuisine_key=cuisine_key,
        cuisine=CUISINE_LABELS.get(cuisine_key, "커피·디저트"),
        place_type="전망 포인트" if kind == "drive" else "장소",
        premium_score=premium,
        premium_signals=["테스트용 등록 메타데이터"] if premium else [],
    )

fixture_restaurants = [Recommendation(fixture_place(i, "restaurant", premium=30 + i * 5), evidence=["식사 예산 탐색 단계: 특별한 식사"], unknowns=["실제 2인 결제금액"]) for i in range(4)]
fixture_cafes = [Recommendation(fixture_place(i, "cafe"), evidence=["카페 순위에는 식사 예산 미적용"]) for i in range(4)]
fixture_drives = [Recommendation(fixture_place(i, "drive"), travel_minutes=20 + i * 6, travel_source="OSRM", evidence=[f"OSRM 도로 이동 {20 + i * 6}분"]) for i in range(4)]

fixture_itinerary = build_itinerary(sample_request, fixture_restaurants, fixture_cafes, fixture_drives)
fixture_checks = verify_result(fixture_restaurants, fixture_cafes, fixture_drives, fixture_itinerary, sample_request)

for check in fixture_checks:
    print(f"{'PASS' if check['passed'] else 'FAIL'}  {check['name']}")
assert all(check["passed"] for check in fixture_checks)


In [ ]:
# 예산이 높아지면 식당의 premium_score 영향은 커지지만 카페 점수에는 변화가 없어야 합니다.
origin = Coordinates(37.54, 127.05)
premium_restaurant = fixture_place(1, "restaurant", premium=80)
regular_restaurant = fixture_place(1, "restaurant", premium=10)
cafe = fixture_place(1, "cafe")

light_request = ExperienceRequest(query="서울", budget_tier="light")
signature_request = ExperienceRequest(query="서울", budget_tier="signature")

restaurant_delta_light = restaurant_score(premium_restaurant, light_request, origin) - restaurant_score(regular_restaurant, light_request, origin)
restaurant_delta_signature = restaurant_score(premium_restaurant, signature_request, origin) - restaurant_score(regular_restaurant, signature_request, origin)
cafe_light = cafe_score(cafe, origin, rainy=False)
cafe_signature = cafe_score(cafe, origin, rainy=False)

assert restaurant_delta_signature > restaurant_delta_light
assert cafe_light == cafe_signature
assert all((item.travel_minutes or 0) >= 20 for item in fixture_drives)
print({
    "higher_budget_increases_premium_restaurant_weight": True,
    "cafe_ignores_restaurant_budget": cafe_light == cafe_signature,
    "all_drives_at_least_20_minutes": True,
})


## 11. 프로덕션 TypeScript·React·Next.js 코드 연결


In [ ]:
PRODUCTION_CODE_MAP = {
    "app/ExperienceAgent.tsx": {
        "stack": ["React 19.2", "TypeScript 5.9"],
        "responsibility": ["조건 입력", "카테고리별 4곳", "메뉴", "동선", "ReAct", "6단계 감사"],
    },
    "app/api/experience/route.ts": {
        "stack": ["Next.js App Router", "TypeScript", "Nominatim", "Open-Meteo", "Overpass", "OSRM"],
        "responsibility": ["도구 오케스트레이션", "후보 정규화", "예산 순위", "20분 검증", "응답"],
    },
    "app/agent/system-prompt.ts": {
        "stack": ["Bounded ReAct", "6-stage reasoning policy"],
        "responsibility": ["진실성 정책", "액션 예산", "FACT/INFERENCE/UNKNOWN"],
    },
    "app/agent/model.ts": {
        "stack": ["GPT-5.6 Sol", "OpenAI Responses API"],
        "responsibility": ["최종 안전 비평", "키 없음·호출 실패 폴백"],
    },
    "vite.config.ts": {
        "stack": ["vinext", "Vite", "Cloudflare plugin"],
        "responsibility": ["Worker 호환 빌드"],
    },
    "db/schema.ts": {
        "stack": ["Drizzle ORM", "Cloudflare D1-ready"],
        "responsibility": ["향후 선호·피드백·저장 계획 확장"],
    },
}

for file_path, info in PRODUCTION_CODE_MAP.items():
    exists = (ROOT / file_path).exists()
    print(f"{'PASS' if exists else 'FAIL'} {file_path:35} {' + '.join(info['stack'])}")
assert all((ROOT / file_path).exists() for file_path in PRODUCTION_CODE_MAP)


In [ ]:
# Notebook이 프로덕션 코드의 핵심 계약과 실제로 연결되어 있는지 확인합니다.
route_source = (ROOT / "app/api/experience/route.ts").read_text(encoding="utf-8")
prompt_source = (ROOT / "app/agent/system-prompt.ts").read_text(encoding="utf-8")
model_source = (ROOT / "app/agent/model.ts").read_text(encoding="utf-8")
ui_source = (ROOT / "app/ExperienceAgent.tsx").read_text(encoding="utf-8")
vite_source = (ROOT / "vite.config.ts").read_text(encoding="utf-8")

SOURCE_CONTRACTS = {
    "GPT-5.6 Sol model id": 'AGENT_MODEL_ID = "gpt-5.6-sol"' in model_source,
    "OpenAI Responses API": "api.openai.com/v1/responses" in model_source,
    "Reasoning effort": 'reasoning: { effort: "medium" }' in model_source,
    "Private reasoning policy": "Do not expose raw chain-of-thought" in prompt_source,
    "Bounded ReAct": "bounded ReAct loop" in prompt_source,
    "Six stages": all(key in prompt_source for key in ["REQUEST", "GROUND", "FILTER", "PLAN", "VERIFY", "REPORT"]),
    "Four candidates": "RECOMMENDATION_LIMIT = 4" in route_source,
    "Drive minimum 20": "MIN_DRIVE_MINUTES = 20" in route_source,
    "OSRM Route": "route/v1" in route_source,
    "OSRM Table": "table/v1/driving" in route_source,
    "Restaurant-only budget": "Never apply it to cafes" in prompt_source,
    "Candidate board UI": "4 PER CATEGORY" in ui_source,
    "vinext Vite plugin": "vinext()" in vite_source,
}

for name, passed in SOURCE_CONTRACTS.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(SOURCE_CONTRACTS.values())


## 12. 보안·환경변수·배포 검증


In [ ]:
REQUIRED_FILES = [
    "README.md", "package.json", ".env.example", ".gitignore",
    ".openai/hosting.json", "app/ExperienceAgent.tsx",
    "app/agent/model.ts", "app/agent/system-prompt.ts",
    "app/api/experience/route.ts", "db/schema.ts",
    "tests/rendered-html.test.mjs", "public/og.png",
]
file_checks = {path: (ROOT / path).exists() for path in REQUIRED_FILES}

gitignore = (ROOT / ".gitignore").read_text(encoding="utf-8")
env_example = (ROOT / ".env.example").read_text(encoding="utf-8")
security_checks = {
    "real env ignored": ".env*" in gitignore,
    "example env allowed": "!.env.example" in gitignore,
    "placeholder only": "your_openai_api_key" in env_example,
    "no real key pattern in example": "sk-" not in env_example,
    "server-side model call": "process.env.OPENAI_API_KEY" in model_source,
}

for name, passed in {**file_checks, **security_checks}.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(file_checks.values()) and all(security_checks.values())


In [ ]:
PIPELINE_COMMANDS = {
    "install": "npm install",
    "development": "npm run dev",
    "type/build": "npm run build",
    "test": "npm test",
    "lint": "npm run lint",
    "database migration": "npm run db:generate",
}
hosting = json.loads((ROOT / ".openai/hosting.json").read_text(encoding="utf-8"))
print("Commands")
for stage, command in PIPELINE_COMMANDS.items():
    print(f"- {stage:20}: {command}")
print("Hosting config loaded:", bool(hosting))
print("Secret required at runtime: OPENAI_API_KEY (value is never printed)")


## 13. 수정 — 사용자 피드백이 코드 계약으로 바뀐 과정

| 피드백 | 구현 변경 | 검증 방식 |
| --- | --- | --- |
| 점심뿐 아니라 저녁·카페·드라이브 전체 동선 | 활동 모델과 순차 일정 생성 | 요청 활동 커버리지 |
| 식당 메뉴를 보고 싶다 | 등록 메뉴 URL + 업종별 가이드 분리 | `registered_menu_link` / `category_guide` |
| 환각을 줄이고 싶다 | 6단계 감사와 FACT/INFERENCE/UNKNOWN | 결정론적 Verify |
| ReAct를 적용하고 싶다 | Action·Observation·Decision·Replan 공개 | bounded action budget |
| 고급 식사 예산 단계 | premium signal 기반 탐색·순위 | 가격을 사실로 단정하지 않음 |
| 2인 예산은 식사금액만 | 식당 점수 함수에만 budget 전달 | 카페 점수 동일성 테스트 |
| 후보를 각 4곳으로 | exact-four 하드 체크 | 개수·중복 assert |
| 드라이브는 20분 이상 | OSRM Table 필터 | 모든 분 값 `>= 20` |
| GPT-5.6-Sol 사용 | Responses API 최종 critic | 실패 시 deterministic fallback |
| 글씨를 키우고 싶다 | React/CSS 정보 계층·대비 개선 | 렌더링 회귀 테스트 |


## 14. 회고

### 잘한 점

- LLM은 장소를 발명하거나 하드 제약을 보장하는 역할이 아니라, 검증된 JSON의 최종 비평만 담당합니다.
- 위치·날씨·장소·경로를 각각 독립 도구로 구성해 실패 원인을 추적할 수 있습니다.
- 카페와 드라이브를 식사 예산에서 분리하고, 드라이브 20분을 코드로 검증합니다.
- ReAct 로그와 6단계 감사로 포트폴리오 평가자가 Agent의 행동과 결과를 확인할 수 있습니다.

### 한계

- OpenStreetMap 등록 정보는 실제 가격·분위기·예약·영업 여부를 보장하지 않습니다.
- OSRM 공개 서버는 실시간 교통을 반영하지 않으며 실패 시 추정치가 표시됩니다.
- 업종별 메뉴 가이드는 실제 메뉴가 아니므로 UI에서 구분해야 합니다.

### 다음 단계

1. 실제 메뉴·예약 API를 연결하고 출처별 최신 시각을 표시합니다.
2. 사용자가 후보를 선택하면 남은 일정을 즉시 다시 계산하는 Human-in-the-loop를 추가합니다.
3. 평가 데이터셋으로 후보 정확성·제약 준수율·도구 실패 복구율을 측정합니다.
4. 선호·피드백을 D1/Drizzle에 저장하되 사용자 동의와 삭제 기능을 함께 설계합니다.


## 공식·외부 기술 문서

- [OpenAI GPT-5.6 Sol](https://developers.openai.com/api/docs/models/gpt-5.6-sol)
- [OpenAI Responses API](https://developers.openai.com/api/docs/guides/responses-vs-chat-completions)
- [OpenAI Function calling](https://developers.openai.com/api/docs/guides/function-calling)
- [OpenStreetMap](https://www.openstreetmap.org/)
- [Nominatim API](https://nominatim.org/release-docs/latest/api/Overview/)
- [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API)
- [Open-Meteo](https://open-meteo.com/en/docs)
- [OSRM API](https://project-osrm.org/docs/v5.24.0/api/)
- [vinext](https://github.com/cloudflare/vinext)

이 Notebook은 학습·포트폴리오용 전체 구성입니다. 프로덕션 구현의 기준 소스는 저장소의 TypeScript 파일이며, Notebook은 그 계약을 자동 검사합니다.
